In [ ]:
import os
import torch
import timm
from tqdm.auto import tqdm
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np

In [ ]:
class ConvBlock(nn.Module):
    """Double conv: (Conv → BN → ReLU) × 2  —  same as Attention U-Net"""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)
class SwinEncoder(nn.Module):
    """
    Wraps SwinV2-Tiny (pretrained ImageNet) as a 4-stage feature extractor.
    Input : (B, 3, 256, 256)
    Output: list of 4 NCHW feature maps
        e1 → (B,  96, 64, 64)   stride-4
        e2 → (B, 192, 32, 32)   stride-8
        e3 → (B, 384, 16, 16)   stride-16
        e4 → (B, 768,  8,  8)   stride-32
    Note: timm returns NHWC tensors — we permute to NCHW here.
    """
    def __init__(self, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(
            'swinv2_tiny_window8_256',
            pretrained=pretrained,
            features_only=True,
            out_indices=(0, 1, 2, 3),
        )
    def forward(self, x):
        feats = self.backbone(x)
        return [f.permute(0, 3, 1, 2).contiguous()
                for f in feats]
class DecoderBlock(nn.Module):
    """
    One decoder step:
        ConvTranspose (upsample ×2) → concat skip → ConvBlock
    in_ch   : channels coming from the deeper layer
    skip_ch : channels of the matching encoder skip connection
    out_ch  : output channels after ConvBlock
    """
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up   = nn.ConvTranspose2d(in_ch, in_ch // 2, kernel_size=2, stride=2)
        self.conv = ConvBlock(in_ch // 2 + skip_ch, out_ch)
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=True)
        x = torch.cat([skip, x], dim=1)
        return self.conv(x)

In [ ]:
class SwinUNet(nn.Module):
    """
    Swin-UNet: SwinV2-Tiny encoder  +  CNN UNet decoder.
    Encoder stages (SwinV2-Tiny, pretrained ImageNet):
        e1: (B,  96, 64, 64)
        e2: (B, 192, 32, 32)
        e3: (B, 384, 16, 16)
        e4: (B, 768,  8,  8)   ← deepest encoder output used as bottleneck
    Decoder:inference
        dec3: up(e4)  + e3  → (B, 384, 16, 16)
        dec2: up(dec3)+ e2  → (B, 192, 32, 32)
        dec1: up(dec2)+ e1  → (B,  96, 64, 64)
        head: upsample ×4   → (B,   1, 256, 256)  raw logits
    Input : (B, 3, 256, 256)   — 3-channel (grayscale repeated to RGB)
    Output: (B, 1, 256, 256)   — raw logits, sigmoid → probability map
    """
    def __init__(self, pretrained=True):
        super().__init__()
        self.encoder = SwinEncoder(pretrained=pretrained)
        self.dec3 = DecoderBlock(in_ch=768, skip_ch=384, out_ch=384)
        self.dec2 = DecoderBlock(in_ch=384, skip_ch=192, out_ch=192)
        self.dec1 = DecoderBlock(in_ch=192, skip_ch=96,  out_ch=96)
        self.head = nn.Sequential(
            nn.Upsample(scale_factor=4, mode='bilinear', align_corners=True),
            nn.Conv2d(96, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 1, kernel_size=1),
        )
    def freeze_encoder(self):
        """Freeze encoder weights — useful for first few warm-up epochs."""
        for p in self.encoder.parameters():
            p.requires_grad = False
    def unfreeze_encoder(self):
        """Unfreeze encoder for full fine-tuning."""
        for p in self.encoder.parameters():
            p.requires_grad = True
    def forward(self, x):
        e1, e2, e3, e4 = self.encoder(x)
        d3 = self.dec3(e4, e3)
        d2 = self.dec2(d3, e2)
        d1 = self.dec1(d2, e1)
        return self.head(d1)

In [ ]:
class FocalTverskyLoss(nn.Module):
    def __init__(self, alpha=0.7, beta=0.3, gamma=0.75, smooth=1e-6):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.smooth = smooth
    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        TP = (probs * targets).sum(dim=(2, 3))
        FP = ((1 - targets) * probs).sum(dim=(2, 3))
        FN = (targets * (1 - probs)).sum(dim=(2, 3))
        Tversky = (TP + self.smooth) / (TP + self.alpha * FP + self.beta * FN + self.smooth)
        FT_loss = (1 - Tversky) ** self.gamma
        return FT_loss.mean()
class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super().__init__()
        self.smooth = smooth
    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        num   = 2 * (probs * targets).sum(dim=(2, 3))
        den   = probs.sum(dim=(2, 3)) + targets.sum(dim=(2, 3)) + self.smooth
        return 1 - (num / den).mean()
class CombinedLoss(nn.Module):
    """Dice + BCE — same as Attention U-Net pipeline"""
    def __init__(self):
        super().__init__()
        self.dice = DiceLoss()
        self.bce  = nn.BCEWithLogitsLoss()
    def forward(self, logits, targets):
        return self.dice(logits, targets) + self.bce(logits, targets)

In [ ]:
class BRISCDataset(Dataset):
    """
    Expects directory structure (same as Attention U-Net pipeline):
        root/
          images/  *.jpg
          masks/   *.png   (same basename as images)
    Images are opened as grayscale then converted to 3-channel RGB
    so pretrained SwinV2 ImageNet weights transfer correctly.
    """
    MEAN = [0.485, 0.456, 0.406]
    STD  = [0.229, 0.224, 0.225]
    def __init__(self, root, img_size=256):
        self.img_dir  = os.path.join(root, "images")
        self.msk_dir  = os.path.join(root, "masks")
        self.names    = sorted(os.listdir(self.img_dir))
        self.img_size = img_size
        self.img_tf = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.Grayscale(num_output_channels=3),
            transforms.ToTensor(),
            transforms.Normalize(mean=self.MEAN, std=self.STD),
        ])
        self.msk_tf = transforms.Compose([
            transforms.Resize((img_size, img_size), interpolation=Image.NEAREST),
            transforms.ToTensor(),
        ])
    def __len__(self):
        return len(self.names)
    def __getitem__(self, idx):
        name     = self.names[idx]
        img_path = os.path.join(self.img_dir, name)
        msk_path = os.path.join(self.msk_dir, os.path.splitext(name)[0] + ".png")
        img  = Image.open(img_path).convert("L")
        mask = Image.open(msk_path).convert("L")
        img  = self.img_tf(img)
        mask = self.msk_tf(mask)
        mask = (mask > 0.5).float()
        return img, mask

In [ ]:
def dice_score(logits, targets, threshold=0.5, smooth=1e-6):
    probs = (torch.sigmoid(logits) > threshold).float()
    num   = 2 * (probs * targets).sum(dim=(2, 3))
    den   = probs.sum(dim=(2, 3)) + targets.sum(dim=(2, 3)) + smooth
    return (num / den).mean().item()
def iou_score(logits, targets, threshold=0.5, smooth=1e-6):
    probs = (torch.sigmoid(logits) > threshold).float()
    inter = (probs * targets).sum(dim=(2, 3))
    union = probs.sum(dim=(2, 3)) + targets.sum(dim=(2, 3)) - inter + smooth
    return (inter / union).mean().item()

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, scaler, device):
    model.train()
    total_loss = 0.0
    pbar = tqdm(loader, desc="Training", leave=False)
    for imgs, masks in pbar:
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            logits = model(imgs)
            loss   = criterion(logits, masks)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})
    return total_loss / len(loader)
@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss, total_dice, total_iou = 0.0, 0.0, 0.0
    pbar = tqdm(loader, desc="Validating", leave=False)
    for imgs, masks in pbar:
        imgs, masks = imgs.to(device), masks.to(device)
        with torch.cuda.amp.autocast():
            logits = model(imgs)
            loss   = criterion(logits, masks)
        total_loss += loss.item()
        total_dice += dice_score(logits, masks)
        total_iou  += iou_score(logits, masks)
    n = len(loader)
    return total_loss / n, total_dice / n, total_iou / n

In [ ]:
DATA_ROOT         = "/home/kartik/Desktop/shalini/brain_brisc/brisc_processed"
SAVE_PATH         = "best_swin_unet.pth"
RESUME_CHECKPOINT = "last_swin_checkpoint.pth"
IMG_SIZE          = 256
BATCH_SIZE        = 8
NUM_EPOCHS        = 4
LR                = 1e-4
WARMUP_EPOCHS     = 8
DEVICE            = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

In [ ]:
train_dir = os.path.join(DATA_ROOT, "train")
val_dir   = os.path.join(DATA_ROOT, "val")
train_ds = BRISCDataset(train_dir, img_size=IMG_SIZE)
val_ds   = BRISCDataset(val_dir,   img_size=IMG_SIZE)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)
print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")

In [ ]:
model     = SwinUNet(pretrained=True).to(DEVICE)
criterion = CombinedLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scaler    = torch.cuda.amp.GradScaler()
best_val_dice = 0.0
start_epoch   = 0
if os.path.exists(RESUME_CHECKPOINT):
    print(f"Loading checkpoint from {RESUME_CHECKPOINT}...")
    checkpoint = torch.load(RESUME_CHECKPOINT, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state"])
    optimizer.load_state_dict(checkpoint["optimizer_state"])
    if "scaler_state" in checkpoint:
        scaler.load_state_dict(checkpoint["scaler_state"])
    start_epoch   = checkpoint["epoch"]
    best_val_dice = checkpoint.get("best_val_dice", 0.0)
    print(f"Resuming from epoch {start_epoch} | Best Val Dice: {best_val_dice:.4f}")
else:
    print("Starting fresh training...")
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=999)
if os.path.exists(RESUME_CHECKPOINT) and "scheduler_state" in checkpoint:
    scheduler.load_state_dict(checkpoint["scheduler_state"])
total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Total parameters: {total_params:.1f}M")

In [ ]:
for epoch in range(start_epoch + 1, start_epoch + NUM_EPOCHS + 1):
    if epoch == 1 and start_epoch == 0:
        model.freeze_encoder()
        print("[Warm-up] Encoder frozen — training decoder only")
    elif epoch == start_epoch + WARMUP_EPOCHS + 1:
        model.unfreeze_encoder()
        print(f"[Epoch {epoch}] Encoder unfrozen — full fine-tuning")
    trn_loss = train_one_epoch(model, train_loader, optimizer, criterion, scaler, DEVICE)
    val_loss, val_dice, val_iou = validate(model, val_loader, criterion, DEVICE)
    scheduler.step()
    print(
        f"Epoch [{epoch:03d}]  "
        f"Train Loss: {trn_loss:.4f}  |  "
        f"Val Loss: {val_loss:.4f}  |  "
        f"Val Dice: {val_dice:.4f}  |  "
        f"Val IoU: {val_iou:.4f}"
    )
    torch.save({
        "epoch":           epoch,
        "model_state":     model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "scaler_state":    scaler.state_dict(),
        "best_val_dice":   best_val_dice,
    }, RESUME_CHECKPOINT)
    if val_dice > best_val_dice:
        best_val_dice = val_dice
        torch.save({
            "epoch":       epoch,
            "model_state": model.state_dict(),
            "val_dice":    val_dice,
            "val_iou":     val_iou,
            "val_loss":    val_loss,
        }, SAVE_PATH)
        print(f"  ✓ Best model saved  (Dice: {best_val_dice:.4f})")
print(f"\nTraining run complete. Best Val Dice overall: {best_val_dice:.4f}")
print(f"Latest checkpoint : {RESUME_CHECKPOINT}")
print(f"Best model saved  : {SAVE_PATH}")